In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.table("medical_catalog.silver.medical_transcriptions_enriched")
print("Total rows:", df.count())
display(df.limit(5))

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_specialty = df.select("medical_specialty").distinct() \
    .withColumn("specialty_id", monotonically_increasing_id())

display(dim_specialty)

dim_specialty.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.dim_specialty")
print("dim_specialty saved!")

In [0]:
dim_sample = df.select("sample_name", "description").distinct() \
    .withColumn("sample_id", monotonically_increasing_id())

display(dim_sample)

dim_sample.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.dim_sample")
print("dim_sample saved!")

In [0]:
dim_diagnosis = df.select("llm_diagnosis", "llm_summary") \
    .filter(col("llm_diagnosis").isNotNull()) \
    .filter(~lower(col("llm_diagnosis")).isin(
        "not specified", "not mentioned", "none", "unknown", "n/a"
    )) \
    .distinct() \
    .withColumn("diagnosis_id", monotonically_increasing_id())

display(dim_diagnosis)

dim_diagnosis.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.dim_diagnosis")
print("dim_diagnosis saved!")

In [0]:
dim_keywords = df \
    .select(explode("keyword_array").alias("keyword")) \
    .withColumn("keyword", lower(trim(col("keyword")))) \
    .filter(col("keyword") != "") \
    .distinct() \
    .withColumn("keyword_id", monotonically_increasing_id())

display(dim_keywords)

dim_keywords.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.dim_keywords")
print("dim_keywords saved!")

In [0]:
fact_medical_cases = df \
    .join(dim_specialty, on="medical_specialty", how="left") \
    .join(dim_sample, on=["sample_name", "description"], how="left") \
    .join(dim_diagnosis, on=["llm_diagnosis", "llm_summary"], how="left") \
    .select(
        col("no").alias("case_id"),
        col("specialty_id"),
        col("sample_id"),
        col("diagnosis_id"),
        col("llm_medications"),
        col("llm_symptoms"),
        col("transcription_word_count"),
        col("keyword_count"),
        col("complexity_bucket"),
        col("has_keywords")
    )

display(fact_medical_cases.limit(10))

fact_medical_cases.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.gold.fact_medical_cases")
print("fact_medical_cases saved!")

In [0]:
#Dim
print("dim_specialty:", spark.table("medical_catalog.gold.dim_specialty").count())
print("dim_sample:", spark.table("medical_catalog.gold.dim_sample").count())
print("dim_diagnosis:", spark.table("medical_catalog.gold.dim_diagnosis").count())
print("dim_keywords:", spark.table("medical_catalog.gold.dim_keywords").count())

#Fact
print("fact_medical_cases:", spark.table("medical_catalog.gold.fact_medical_cases").count())